# 05 — Gold Aggregations

**Week:** 7  
**Date:** September 1–7, 2026

**Goal:** Create dashboard-ready Gold tables and document KPI formulas and metric grain.

The Gold layer is built from the validated ShipTrack Silver tables already available in Databricks.


## Gold tables created

1. `gold_shipment_daily_metrics` — daily shipment KPIs
2. `gold_carrier_metrics` — carrier-level performance
3. `gold_route_metrics` — route-level performance
4. `gold_hub_metrics` — hub-level shipment flow

All Gold tables are aggregated from Silver data and are intended for dashboard and Power BI use.


In [ ]:
%sql

SHOW TABLES;


In [ ]:
%sql

CREATE OR REPLACE TABLE gold_shipment_daily_metrics AS
SELECT
    CAST(booking_ts AS DATE) AS metric_date,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN actual_delivery_ts IS NOT NULL
                THEN (unix_timestamp(actual_delivery_ts) - unix_timestamp(pickup_ts)) / 3600.0
            END
        ),
        2
    ) AS avg_delivery_hours,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM silver_shipments
GROUP BY CAST(booking_ts AS DATE);


In [ ]:
%sql

SELECT *
FROM gold_shipment_daily_metrics
ORDER BY metric_date;


In [ ]:
%sql

CREATE OR REPLACE TABLE gold_carrier_metrics AS
SELECT
    carrier_id,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN actual_delivery_ts IS NOT NULL
                THEN (unix_timestamp(actual_delivery_ts) - unix_timestamp(pickup_ts)) / 3600.0
            END
        ),
        2
    ) AS avg_delivery_hours,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM silver_shipments
GROUP BY carrier_id;


In [ ]:
%sql

SELECT *
FROM gold_carrier_metrics
ORDER BY total_shipments DESC;


In [ ]:
%sql

CREATE OR REPLACE TABLE gold_route_metrics AS
SELECT
    route_id,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(AVG(package_weight_kg), 2) AS avg_package_weight_kg,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM silver_shipments
GROUP BY route_id;


In [ ]:
%sql

SELECT *
FROM gold_route_metrics
ORDER BY total_shipments DESC;


In [ ]:
%sql

CREATE OR REPLACE TABLE gold_hub_metrics AS
WITH origin_metrics AS (
    SELECT
        origin_hub_id AS hub_id,
        COUNT(*) AS origin_shipments,
        0 AS destination_shipments,
        SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS origin_delivered
    FROM silver_shipments
    GROUP BY origin_hub_id
),
destination_metrics AS (
    SELECT
        destination_hub_id AS hub_id,
        0 AS origin_shipments,
        COUNT(*) AS destination_shipments,
        SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS destination_delivered
    FROM silver_shipments
    GROUP BY destination_hub_id
),
combined AS (
    SELECT * FROM origin_metrics
    UNION ALL
    SELECT * FROM destination_metrics
)
SELECT
    hub_id,
    SUM(origin_shipments) AS origin_shipments,
    SUM(destination_shipments) AS destination_shipments,
    SUM(origin_delivered) AS delivered_shipments,
    SUM(origin_shipments) + SUM(destination_shipments) AS total_hub_shipments
FROM combined
GROUP BY hub_id;


In [ ]:
%sql

SELECT
    h.hub_id,
    h.hub_name,
    h.city,
    h.region,
    g.origin_shipments,
    g.destination_shipments,
    g.delivered_shipments,
    g.total_hub_shipments
FROM gold_hub_metrics g
LEFT JOIN silver_hubs h
    ON g.hub_id = h.hub_id
ORDER BY total_hub_shipments DESC;


## Gold table validation

The following checks verify that the Gold tables were created successfully, have reasonable row counts, and do not contain unexpected nulls in their key metric fields.


In [ ]:
%sql

SELECT
    'gold_shipment_daily_metrics' AS table_name,
    COUNT(*) AS row_count
FROM gold_shipment_daily_metrics

UNION ALL

SELECT
    'gold_carrier_metrics',
    COUNT(*)
FROM gold_carrier_metrics

UNION ALL

SELECT
    'gold_route_metrics',
    COUNT(*)
FROM gold_route_metrics

UNION ALL

SELECT
    'gold_hub_metrics',
    COUNT(*)
FROM gold_hub_metrics;


In [ ]:
%sql

SELECT
    COUNT(*) AS invalid_rows
FROM gold_shipment_daily_metrics
WHERE metric_date IS NULL
   OR total_shipments < 0
   OR delivery_rate_pct < 0
   OR delivery_rate_pct > 100
   OR on_time_delivery_rate_pct < 0
   OR on_time_delivery_rate_pct > 100;


In [ ]:
%sql

SELECT
    COUNT(*) AS invalid_rows
FROM gold_carrier_metrics
WHERE carrier_id IS NULL
   OR total_shipments < 0
   OR delivery_rate_pct < 0
   OR delivery_rate_pct > 100
   OR on_time_delivery_rate_pct < 0
   OR on_time_delivery_rate_pct > 100;


In [ ]:
%sql

SELECT
    COUNT(*) AS invalid_rows
FROM gold_route_metrics
WHERE route_id IS NULL
   OR total_shipments < 0
   OR delivery_rate_pct < 0
   OR delivery_rate_pct > 100
   OR on_time_delivery_rate_pct < 0
   OR on_time_delivery_rate_pct > 100;


In [ ]:
%sql

SELECT
    COUNT(*) AS invalid_rows
FROM gold_hub_metrics
WHERE hub_id IS NULL
   OR total_hub_shipments < 0;


## Dashboard-ready output

The four Gold tables created above are intended to be the only datasets used for the Week 7 dashboard layer.

Recommended dashboard views:

- Overview: `gold_shipment_daily_metrics`
- Carrier performance: `gold_carrier_metrics`
- Route performance: `gold_route_metrics`
- Hub flow: `gold_hub_metrics`

KPI formulas are documented in `docs/gold_metrics_definition.md`.


## Week 7 completion checklist

- Gold tables created
- KPI calculations implemented
- Metric grain defined
- Gold tables validated
- Outputs are suitable for Power BI/dashboard use
- KPI definitions documented separately
